# AVIRSO — essai : une tête qui tourne à partir d'une seule photo

Ce carnet teste le modèle **LivePortrait** (code et poids sous licence MIT). À partir d'une photo de face, il génère la même tête vue de −30° à +30°.

**Comment faire** : menu *Exécution* → *Tout exécuter*. Comptez quelques minutes. Le résultat s'affiche tout en bas.

- Par défaut, l'essai utilise une **photo de test** (visage fictif).
- Pour essayer avec votre propre photo, mettez `MA_PHOTO = True` dans la deuxième cellule : la photo est alors envoyée sur les serveurs de Google (Colab) le temps de l'essai.
- Les images produites sont **générées par un modèle** : ce ne sont pas des photos.

In [ ]:
# 1. Installation : le projet et ses poids (environ 520 Mo)
!git clone -q --depth 1 https://github.com/KwaiVGI/LivePortrait
%cd LivePortrait
!mkdir -p pw sortie
B = 'https://huggingface.co/KwaiVGI/LivePortrait/resolve/main/liveportrait'
for f in ['base_models/appearance_feature_extractor.pth', 'base_models/motion_extractor.pth', 'base_models/spade_generator.pth', 'base_models/warping_module.pth', 'retargeting_models/stitching_retargeting_module.pth']:
    !wget -q -nc -P pw {B}/{f}
!ls -la pw

In [ ]:
# 2. La photo : photo de test, ou la vôtre
MA_PHOTO = False
if MA_PHOTO:
    from google.colab import files
    envoi = files.upload()
    PHOTO = list(envoi.keys())[0]
else:
    !wget -q -O photo_test.jpg https://raw.githubusercontent.com/Copsenova/avirso-test/main/essai-generation/photo_test.jpg
    PHOTO = 'photo_test.jpg'
print('photo utilisée :', PHOTO)

In [ ]:
%%writefile rotation.py
# Essai de LivePortrait (code et poids sous licence MIT) sur CPU.
# But : à partir d'UNE photo de face, produire la même tête vue sous d'autres angles, pour juger la qualité et le temps de calcul.
# Photo de test uniquement. Le cadrage du visage est fait avec OpenCV.
import sys, time, yaml, numpy as np, cv2, torch
from src.utils.helper import load_model
from src.utils.camera import headpose_pred_to_degree, get_rotation_matrix

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('calcul sur :', dev)
cfg = yaml.safe_load(open('src/config/models.yaml'))
F = load_model('pw/appearance_feature_extractor.pth', cfg, dev, 'appearance_feature_extractor')
M = load_model('pw/motion_extractor.pth', cfg, dev, 'motion_extractor')
W = load_model('pw/warping_module.pth', cfg, dev, 'warping_module')
G = load_model('pw/spade_generator.pth', cfg, dev, 'spade_generator')
S = load_model('pw/stitching_retargeting_module.pth', cfg, dev, 'stitching_retargeting_module')

src, out = sys.argv[1], sys.argv[2]
pas = float(sys.argv[3]) if len(sys.argv) > 3 else 5

img = cv2.imread(src)
gris = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
detecteur = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
boites = detecteur.detectMultiScale(gris, 1.1, 5, minSize=(80, 80))
assert len(boites), 'visage non détecté'
x, y, w, h = max(boites, key=lambda b: b[2] * b[3])
cote = int(2.05 * w)
cx, cy = x + w / 2, y + h / 2 - 0.02 * cote
x0, y0 = int(cx - cote / 2), int(cy - cote / 2)
bord = cv2.copyMakeBorder(img, cote, cote, cote, cote, cv2.BORDER_REPLICATE)
toile = bord[y0 + cote:y0 + 2 * cote, x0 + cote:x0 + 2 * cote]
rgb = cv2.cvtColor(cv2.resize(toile, (256, 256), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
cv2.imwrite(out + '_source.png', cv2.resize(toile, (512, 512), interpolation=cv2.INTER_CUBIC))
xs = torch.from_numpy(rgb.astype(np.float32) / 255).permute(2, 0, 1)[None].to(dev)

with torch.no_grad():
    t0 = time.time()
    fs = F(xs)
    k = M(xs)
    bs = 1
    pitch, yaw, roll = [headpose_pred_to_degree(k[n])[:, None] for n in ('pitch', 'yaw', 'roll')]
    kp = k['kp'].reshape(bs, -1, 3)
    ex = k['exp'].reshape(bs, -1, 3)
    sc = k['scale']
    tr = k['t'].clone()
    tr[:, 2] = 0

    def points(dy):
        R = get_rotation_matrix(pitch, yaw + dy, roll)
        p = (kp @ R + ex) * sc[..., None]
        p[:, :, 0:2] += tr[:, None, 0:2]
        return p

    ks = points(0.0)
    print('pose de la photo (tangage, lacet, roulis) :', [round(float(v), 1) for v in (pitch, yaw, roll)], '| préparation %.1f s' % (time.time() - t0))
    angles = np.arange(-30, 30.01, pas)
    durees = []
    n = kp.shape[1]
    for i, a in enumerate(angles):
        t0 = time.time()
        kd = points(float(a))
        d = S['stitching'](torch.cat([ks.reshape(bs, -1), kd.reshape(bs, -1)], 1))
        kd = kd + d[..., :3 * n].reshape(bs, n, 3)
        kd[..., :2] += d[..., 3 * n:3 * n + 2].reshape(bs, 1, 2)
        im = G(feature=W(fs, kp_source=ks, kp_driving=kd)['out'])
        durees.append(time.time() - t0)
        sortie = (im[0].permute(1, 2, 0).clamp(0, 1).float().cpu().numpy() * 255).astype(np.uint8)
        cv2.imwrite(f'{out}_{i:03d}.png', cv2.cvtColor(sortie, cv2.COLOR_RGB2BGR))

print(f'{len(angles)} images, {np.mean(durees):.2f} s par image (max {np.max(durees):.2f}) sur {dev}')


In [ ]:
# 3. Génération : une image tous les 2,5 degrés, de −30° à +30°
!python rotation.py "{PHOTO}" sortie/t 2.5

In [ ]:
# 4. Résultat : l'animation, puis la photo d'origine à côté de trois angles générés
import glob, imageio.v2 as imageio, numpy as np
from IPython.display import Image as Montrer, display
from PIL import Image
imgs = [imageio.imread(f) for f in sorted(glob.glob('sortie/t_[0-9]*.png'))]
imageio.mimsave('rotation.gif', imgs + imgs[::-1], duration=0.07, loop=0)
display(Montrer(filename='rotation.gif'))
n = len(imgs)
planche = np.concatenate([imageio.imread('sortie/t_source.png')[:, :, :3]] + [imgs[i] for i in (0, n // 2, n - 1)], axis=1)
Image.fromarray(planche).save('planche.png')
display(Montrer(filename='planche.png'))
print('À gauche : la photo d\'origine. Ensuite : −30°, 0°, +30°, générés.')